In [ ]:
model_name = "ckiplab/albert-base-chinese-ner"
pretrained_model = "ckiplab/albert-base-chinese"

In [ ]:
!pip install datasets
!pip install transformers
!pip install seqeval -q

## **prepocessing**

In [ ]:
import pandas as pd

train_data = pd.read_json("train.json", lines=True)
valid_data = pd.read_json("test.json", lines=True)
test_data = pd.read_json("conference_test.json")

In [ ]:
data = pd.concat([train_data, valid_data])


In [ ]:
from sklearn.model_selection import train_test_split
train_data, valid_data = train_test_split(data, test_size = 0.09, random_state=0)

In [ ]:
def get(data):
  data = data.loc[:,"character	character_label".split()]
  data.columns = ["tokens","ner_tags"]
  return data
train_data = get(train_data)
valid_data = get(valid_data)
test_data = get(test_data)

In [ ]:
def get_tag_name(df):
  tag_name = []
  assert isinstance(df, pd.DataFrame), #'輸入要是pd.Series，也就是dataframe的ner_tags欄位'
  for i in df.ner_tags:
    tag_name += [e for e in list(set(i)) if not e in tag_name]
  return tag_name
tag_name = get_tag_name(train_data)

In [ ]:
from datasets import Dataset, ClassLabel, Sequence, Features, Value, DatasetDict

tags = ClassLabel(num_classes=len(tag_name), names=tag_name)

In [ ]:
dataset_structure = {"ner_tags":Sequence(tags),
                 'tokens': Sequence(feature=Value(dtype='string'))}

In [ ]:
def df_to_dataset(df, columns=['ner_tags', 'tokens']):
  assert set(['ner_tags', 'tokens']).issubset(df.columns)

  ner_tags = df['ner_tags'].map(tags.str2int).values.tolist()
  tokens = df['tokens'].values.tolist()

  assert isinstance(tokens[0], list) 
  assert isinstance(ner_tags[0], list)
  d = {'ner_tags':ner_tags, 'tokens':tokens}# 如果有其他欄位例如id, spans請從這裡添加
  # create dataset
  dataset = Dataset.from_dict(mapping=d,
              features=Features(dataset_structure),)
  return dataset

train_dataset = df_to_dataset(train_data) # 從train.txt變成df，然後轉成訓練資料dataset
valid_dataset =  df_to_dataset(valid_data) # 從test-submit.txt變成test_df，然後轉成訓練資料test_dataset
test_dataset = df_to_dataset(test_data)


# gather everyone if you want to have a single DatasetDict
dataset = DatasetDict({
    'train': train_dataset, # Trainer會用到
    'test':  test_dataset, # 獨立的測試資料
    'valid': valid_dataset}) # Trainer會用到

label_names = dataset["train"].features["ner_tags"].feature.names

In [ ]:
from torch.nn import Embedding
from transformers import (
   BertTokenizerFast,
   AutoModelForTokenClassification
)
tokenizer = BertTokenizerFast.from_pretrained(pretrained_model)  #tokenization
def tokenize_function(examples):
    return tokenizer(examples["tokens"], padding="max_length",
                     truncation=True, is_split_into_words=True)


In [ ]:
tokenized_datasets_ = dataset.map(tokenize_function, batched=True)

In [ ]:
#Get the values for input_ids, attention_mask, adjusted labels
def tokenize_adjust_labels(all_samples_per_split):
  tokenized_samples = tokenizer.batch_encode_plus(all_samples_per_split["tokens"],
                is_split_into_words=True, truncation=True)
  # print(tokenized_samples['input_ids'][:2])
  # tokenizer(string, padding=True, truncation=True) 
  # assert False
  print(len(tokenized_samples["input_ids"]))
  print(tokenized_samples.word_ids(batch_index=2))
  total_adjusted_labels = []
  
  for k in range(0, len(tokenized_samples["input_ids"])):
    prev_wid = -1
    word_ids_list = tokenized_samples.word_ids(batch_index=k)
    existing_label_ids = all_samples_per_split["ner_tags"][k]
    i = -1
    adjusted_label_ids = []
    # print(existing_label_ids)
    # print(adjusted_label_ids)
    # assert False
    for word_idx in word_ids_list:
      # Special tokens have a word id that is None. We set the label to -100 so they are automatically
      # ignored in the loss function.
      if(word_idx is None):
        adjusted_label_ids.append(-100)
      elif(word_idx!=prev_wid):
        i = i + 1
        adjusted_label_ids.append(existing_label_ids[i])
        prev_wid = word_idx
      else:
        label_name = label_names[existing_label_ids[i]]
        adjusted_label_ids.append(existing_label_ids[i])
        
    total_adjusted_labels.append(adjusted_label_ids)
  
  #add adjusted labels to the tokenized samples
  tokenized_samples["labels"] = total_adjusted_labels
  return tokenized_samples

tokenized_dataset = dataset.map(tokenize_adjust_labels,
                batched=True,
                remove_columns=list(dataset["train"].features.keys()))

In [ ]:
tokenized_dataset

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AdamW

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(model_name,num_labels=len(label_names),ignore_mismatched_sizes=True) #model selection

# **compute_metrics**

In [ ]:
import numpy as np
from datasets import load_metric
metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p

    #select predicted index with maximum logit for each token
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

# **first training**

In [ ]:
from transformers import TrainingArguments, Trainer

batch_size = 16                #hyperparameter

training_args = TrainingArguments(
    output_dir="/content/",
    evaluation_strategy = "epoch",
    learning_rate=0.00005,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    #weight_decay=0.01,
    push_to_hub=False)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(predictions, axis=2)
# Remove ignored index (special tokens)
true_predictions = [[label_names[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

In [ ]:
true_labels = [
     [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
     for prediction, label in zip(predictions, labels)
 ]
results = metric.compute(predictions=true_predictions, references=true_labels)
results
